In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from pathlib import Path
import time
import json
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')


In [ ]:
# Hyperparameters matching ViT notebook configuration
HYPERPARAMS = {
    'input_shape': (160, 160, 3),
    'num_classes': 12,  # All 12 classes including No-Anomaly
    'batch_size': 8,
    'epochs': 60,
    'learning_rate': 1e-4,
    'learning_rate_decay': 0.2,
    'patience_lr': 5,
    'patience_early_stop': 9,
    'validation_split': 0.1,
    'test_split': 0.1,
    'label_smoothing': 0.2,
    'epsilon': 1e-6,
    'weight_decay': 0.01
}

# Extract commonly used variables
image_size = HYPERPARAMS['input_shape'][0]  # 160
batch_size = HYPERPARAMS['batch_size']  # 8

# Data transforms (matching ViT configuration exactly)
train_transform = transforms.Compose([
    transforms.Resize((HYPERPARAMS['input_shape'][0], HYPERPARAMS['input_shape'][1])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((HYPERPARAMS['input_shape'][0], HYPERPARAMS['input_shape'][1])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(f"Image size: {image_size}x{image_size}")
print(f"Batch size: {batch_size}")
print(f"\n12-Class Hyperparameters:")
for key, value in HYPERPARAMS.items():
    print(f"  {key}: {value}")


In [ ]:
# Dataset paths
data_dir = Path('datasets_paper/12_class_dataset')

# Class names (11 anomaly types + 1 normal)
class_names = [
    'Cell', 'Cell-Multi', 'Cracking', 'Diode', 'Diode-Multi',
    'Hot-Spot', 'Hot-Spot-Multi', 'No-Anomaly', 'Offline-Module', 
    'Shadowing', 'Soiling', 'Vegetation'
]
num_classes = len(class_names)

print(f"Dataset directory: {data_dir}")
print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")

# Count images in each class
total_images = 0
class_distribution = {}
for class_name in class_names:
    class_path = data_dir / class_name
    if class_path.exists():
        num_images = len(list(class_path.glob('*.jpg')))
        class_distribution[class_name] = num_images
        total_images += num_images
        print(f"{class_name}: {num_images} images")
    else:
        print(f"Warning: {class_path} does not exist")

print(f"\nTotal images: {total_images}")

# Visualize class distribution
plt.figure(figsize=(14, 6))
colors = ['red' if name != 'No-Anomaly' else 'green' for name in class_distribution.keys()]
bars = plt.bar(class_distribution.keys(), class_distribution.values(), color=colors, alpha=0.7)
plt.title('12-Class Dataset Distribution (Green = Normal, Red = Anomalies)')
plt.xlabel('Class')
plt.ylabel('Number of Images')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Check class balance
anomaly_count = sum(count for name, count in class_distribution.items() if name != 'No-Anomaly')
normal_count = class_distribution.get('No-Anomaly', 0)
print(f"\nClass Balance:")
print(f"Total Anomaly samples: {anomaly_count}")
print(f"No-Anomaly samples: {normal_count}")
print(f"Anomaly:Normal ratio = {anomaly_count/normal_count:.2f}:1")


In [ ]:
# DeiT Classifier matching ViT architecture
class DeiTClassifier(nn.Module):
    def __init__(self, num_classes=12, pretrained=True, dropout_rate=0.3):
        super(DeiTClassifier, self).__init__()
        
        # Load DeiT-B/16 model (closest to ViT-B/32 architecture)
        self.backbone = timm.create_model(
            'deit_base_patch16_224',  # DeiT-B/16 
            pretrained=pretrained,
            num_classes=0,  # Remove head, we'll add custom one
            img_size=160    # Match ViT input size
        )
        
        # Get the feature dimension (768 for DeiT-B)
        self.feature_dim = self.backbone.num_features
        
        # Custom classification head (matching ViT architecture exactly)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, 0, 0.01)
                torch.nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Extract features using DeiT backbone
        features = self.backbone(x)
        # Classification
        output = self.classifier(features)
        return output

# Selected model configuration
selected_model = 'deit_base_patch16_224'  # DeiT-B/16 (closest equivalent to ViT-B/32)
print(f"Selected model: {selected_model}")
print(f"Note: Using DeiT-B/16 with 160x160 input (closest to ViT-B/32 configuration)")

model = DeiTClassifier(num_classes=HYPERPARAMS['num_classes'], pretrained=True)
model = model.to(device)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Summary:")
print(f"Architecture: DeiT-B/16 with custom head")
print(f"Input size: {HYPERPARAMS['input_shape']}")
print(f"Feature dimension: {model.feature_dim}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: {total_params * 4 / 1024**2:.2f} MB (fp32)")


In [ ]:
# Data Loading and Dataset Preparation
full_dataset = ImageFolder(root=data_dir, transform=train_transform)

# Calculate dataset splits
total_size = len(full_dataset)
test_size = int(HYPERPARAMS['test_split'] * total_size)
val_size = int(HYPERPARAMS['validation_split'] * total_size)
train_size = total_size - test_size - val_size

print(f"Dataset splits:")
print(f"  Train: {train_size} samples ({train_size/total_size*100:.1f}%)")
print(f"  Validation: {val_size} samples ({val_size/total_size*100:.1f}%)")
print(f"  Test: {test_size} samples ({test_size/total_size*100:.1f}%)")

# Split dataset
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply different transforms for validation and test sets
val_dataset.dataset = ImageFolder(root=data_dir, transform=val_transform)
test_dataset.dataset = ImageFolder(root=data_dir, transform=val_transform)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"\nData loaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Create directories for saving models and results
MODEL_SAVE_PATH = Path('models/12_class_deit')
RESULTS_PATH = Path('results/12_class_deit')
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_PATH.mkdir(parents=True, exist_ok=True)


In [ ]:
# Label Smoothing CrossEntropy Loss
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing
    
    def forward(self, pred, target):
        confidence = 1. - self.smoothing
        logprobs = torch.nn.functional.log_softmax(pred, dim=-1)
        nll_loss = -logprobs.gather(dim=-1, index=target.unsqueeze(1))
        nll_loss = nll_loss.squeeze(1)
        smooth_loss = -logprobs.mean(dim=-1)
        loss = confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()

# Loss function and optimizer
criterion = LabelSmoothingCrossEntropy(smoothing=HYPERPARAMS['label_smoothing'])
optimizer = optim.AdamW(
    model.parameters(),
    lr=HYPERPARAMS['learning_rate'],
    weight_decay=HYPERPARAMS['weight_decay'],
    eps=HYPERPARAMS['epsilon']
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=HYPERPARAMS['learning_rate_decay'],
    patience=HYPERPARAMS['patience_lr'],
    verbose=True
)

print("🎯 DeiT Training Configuration:")
print(f"  Loss: CrossEntropy with Label Smoothing ({HYPERPARAMS['label_smoothing']})")
print(f"  Optimizer: AdamW (lr={HYPERPARAMS['learning_rate']}, weight_decay={HYPERPARAMS['weight_decay']})")
print(f"  Scheduler: ReduceLROnPlateau (factor={HYPERPARAMS['learning_rate_decay']}, patience={HYPERPARAMS['patience_lr']})")
print(f"  Early stopping patience: {HYPERPARAMS['patience_early_stop']} epochs")


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    
    for batch_idx, (inputs, targets) in enumerate(progress_bar):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'Loss': f'{running_loss/(batch_idx+1):.4f}',
            'Acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc='Validation', leave=False)
        
        for batch_idx, (inputs, targets) in enumerate(progress_bar):
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            progress_bar.set_postfix({
                'Loss': f'{running_loss/(batch_idx+1):.4f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

print("✅ Training and validation functions defined")
print("Ready to start training DeiT model...")


In [8]:
# Training loop with early stopping
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, 
                num_epochs, patience, device, save_path):
    """Complete training loop with early stopping"""
    
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    patience_counter = 0
    
    print(f"🚀 Starting DeiT Training for {num_epochs} epochs")
    print(f"📊 Model: {model.__class__.__name__}")
    print(f"💾 Best model will be saved to: {save_path}")
    print("-" * 60)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        epoch_start = time.time()
        
        # Train and validate
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_epoch(model, val_loader, criterion, device)
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        epoch_time = time.time() - epoch_start
        
        # Print epoch results
        print(f"Epoch {epoch+1:2d}/{num_epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:6.2f}% | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:6.2f}% | "
              f"Time: {epoch_time:.1f}s")
        
        # Early stopping and model saving
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            patience_counter = 0
            
            # Save best model
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_loss': best_val_loss,
                'best_val_acc': best_val_acc,
                'hyperparams': HYPERPARAMS,
                'class_names': class_names
            }, save_path / 'best_model_12class_deit.pth')
            
            print(f"💾 New best model saved! Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            print(f"\n⏹️  Early stopping triggered after {epoch+1} epochs")
            print(f"🏆 Best validation accuracy: {best_val_acc:.2f}%")
            break
    
    total_time = time.time() - start_time
    print(f"\n🏁 Training completed in {total_time/60:.1f} minutes")
    print(f"🏆 Best validation loss: {best_val_loss:.4f}")
    print(f"🏆 Best validation accuracy: {best_val_acc:.2f}%")
    
    return history

# Start training
print(f"💡 Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"💡 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

training_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=HYPERPARAMS['epochs'],
    patience=HYPERPARAMS['patience_early_stop'],
    device=device,
    save_path=MODEL_SAVE_PATH
)


KeyboardInterrupt: 

In [ ]:
# Plot training history
def plot_training_history(history, save_path=None):
    """Plot training and validation metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('DeiT 12-Class Training History', fontsize=16, fontweight='bold')
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss plots
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Training Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Validation Loss', linewidth=2)
    axes[0, 0].set_title('Model Loss', fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy plots
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Training Accuracy', linewidth=2)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Validation Accuracy', linewidth=2)
    axes[0, 1].set_title('Model Accuracy', fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Loss difference
    loss_diff = [abs(t - v) for t, v in zip(history['train_loss'], history['val_loss'])]
    axes[1, 0].plot(epochs, loss_diff, 'g-', linewidth=2)
    axes[1, 0].set_title('Train-Validation Loss Difference', fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('|Train Loss - Val Loss|')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Accuracy difference
    acc_diff = [abs(t - v) for t, v in zip(history['train_acc'], history['val_acc'])]
    axes[1, 1].plot(epochs, acc_diff, 'purple', linewidth=2)
    axes[1, 1].set_title('Train-Validation Accuracy Difference', fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('|Train Acc - Val Acc| (%)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path / 'training_history_12class_deit.png', dpi=300, bbox_inches='tight')
        print(f"📊 Training history plot saved to: {save_path}")
    
    plt.show()
    
    # Print final metrics
    print("📈 Final Training Metrics:")
    print(f"  Final Train Loss: {history['train_loss'][-1]:.4f}")
    print(f"  Final Train Accuracy: {history['train_acc'][-1]:.2f}%")
    print(f"  Final Val Loss: {history['val_loss'][-1]:.4f}")
    print(f"  Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
    print(f"  Best Val Loss: {min(history['val_loss']):.4f}")
    print(f"  Best Val Accuracy: {max(history['val_acc']):.2f}%")

# Plot training history
plot_training_history(training_history, RESULTS_PATH)


In [ ]:
# Load best model and evaluate on test set
def load_best_model(model_path, model, device):
    """Load the best saved model"""
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✅ Best model loaded from epoch {checkpoint['epoch']}")
    print(f"📊 Best validation accuracy: {checkpoint['best_val_acc']:.2f}%")
    return model

def evaluate_model(model, test_loader, device, class_names):
    """Comprehensive model evaluation"""
    model.eval()
    all_predictions = []
    all_targets = []
    
    print("🧪 Evaluating model on test set...")
    
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc='Testing'):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_targets, all_predictions)
    precision, recall, f1, support = precision_recall_fscore_support(
        all_targets, all_predictions, average=None, zero_division=0
    )
    
    # Overall metrics
    precision_macro = precision_recall_fscore_support(all_targets, all_predictions, average='macro', zero_division=0)[0]
    recall_macro = precision_recall_fscore_support(all_targets, all_predictions, average='macro', zero_division=0)[1]
    f1_macro = precision_recall_fscore_support(all_targets, all_predictions, average='macro', zero_division=0)[2]
    
    precision_weighted = precision_recall_fscore_support(all_targets, all_predictions, average='weighted', zero_division=0)[0]
    recall_weighted = precision_recall_fscore_support(all_targets, all_predictions, average='weighted', zero_division=0)[1]
    f1_weighted = precision_recall_fscore_support(all_targets, all_predictions, average='weighted', zero_division=0)[2]
    
    print(f"\n🎯 DeiT 12-Class Test Results:")
    print(f"Overall Accuracy: {accuracy*100:.2f}%")
    print(f"Macro Avg - Precision: {precision_macro:.4f}, Recall: {recall_macro:.4f}, F1: {f1_macro:.4f}")
    print(f"Weighted Avg - Precision: {precision_weighted:.4f}, Recall: {recall_weighted:.4f}, F1: {f1_weighted:.4f}")
    
    # Per-class results
    print(f"\n📊 Per-Class Results:")
    print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10} {'Accuracy':<10}")
    print("-" * 75)
    
    for i, class_name in enumerate(class_names):
        class_accuracy = (np.array(all_targets) == i).sum() > 0 and \
                        ((np.array(all_predictions) == i) & (np.array(all_targets) == i)).sum() / (np.array(all_targets) == i).sum() * 100
        print(f"{class_name:<15} {precision[i]:<10.3f} {recall[i]:<10.3f} {f1[i]:<10.3f} {support[i]:<10} {class_accuracy:<10.1f}%")
    
    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'per_class_precision': precision,
        'per_class_recall': recall,
        'per_class_f1': f1,
        'per_class_support': support,
        'predictions': all_predictions,
        'targets': all_targets
    }

# Load best model and evaluate
model = load_best_model(MODEL_SAVE_PATH / 'best_model_12class_deit.pth', model, device)
test_results = evaluate_model(model, test_loader, device, class_names)


In [ ]:
# Confusion Matrix Visualization
def plot_confusion_matrix(y_true, y_pred, class_names, save_path=None):
    """Plot confusion matrix with detailed analysis"""
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = confusion_matrix(y_true, y_pred, normalize='true')
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Raw confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[0], cbar_kws={'label': 'Count'})
    axes[0].set_title('DeiT 12-Class Confusion Matrix (Raw Counts)', fontweight='bold', fontsize=14)
    axes[0].set_xlabel('Predicted Class', fontweight='bold')
    axes[0].set_ylabel('True Class', fontweight='bold')
    
    # Normalized confusion matrix
    sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=axes[1], cbar_kws={'label': 'Proportion'})
    axes[1].set_title('DeiT 12-Class Confusion Matrix (Normalized)', fontweight='bold', fontsize=14)
    axes[1].set_xlabel('Predicted Class', fontweight='bold')
    axes[1].set_ylabel('True Class', fontweight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path / 'confusion_matrix_12class_deit.png', dpi=300, bbox_inches='tight')
        print(f"📊 Confusion matrix saved to: {save_path}")
    
    plt.show()
    
    return cm, cm_normalized

def analyze_classification_errors(y_true, y_pred, class_names):
    """Analyze classification errors in detail"""
    cm = confusion_matrix(y_true, y_pred)
    
    print("🔍 Classification Error Analysis:")
    print("="*60)
    
    # Find most confused pairs
    confused_pairs = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i][j] > 0:
                confused_pairs.append((class_names[i], class_names[j], cm[i][j]))
    
    # Sort by confusion count
    confused_pairs.sort(key=lambda x: x[2], reverse=True)
    
    print("Top 10 Most Confused Class Pairs:")
    print(f"{'True Class':<15} {'→ Predicted As':<15} {'Count':<8} {'Description'}")
    print("-" * 70)
    
    for i, (true_class, pred_class, count) in enumerate(confused_pairs[:10]):
        if i < 10:
            description = "Anomaly→Normal" if pred_class == "No-Anomaly" and true_class != "No-Anomaly" else \
                         "Normal→Anomaly" if true_class == "No-Anomaly" and pred_class != "No-Anomaly" else \
                         "Anomaly→Anomaly"
            print(f"{true_class:<15} → {pred_class:<15} {count:<8} {description}")
    
    # Analyze anomaly vs normal confusion
    anomaly_classes = [name for name in class_names if name != 'No-Anomaly']
    normal_class = 'No-Anomaly'
    normal_idx = class_names.index(normal_class)
    
    # False positives: Normal classified as anomaly
    false_positives = sum(cm[normal_idx][i] for i in range(len(class_names)) if i != normal_idx)
    
    # False negatives: Anomaly classified as normal  
    false_negatives = sum(cm[i][normal_idx] for i in range(len(class_names)) if i != normal_idx)
    
    total_normal = sum(cm[normal_idx])
    total_anomaly = sum(sum(cm[i]) for i in range(len(class_names)) if i != normal_idx)
    
    print(f"\n🚨 Anomaly Detection Analysis:")
    print(f"False Positives (Normal→Anomaly): {false_positives}/{total_normal} ({false_positives/total_normal*100:.1f}%)")
    print(f"False Negatives (Anomaly→Normal): {false_negatives}/{total_anomaly} ({false_negatives/total_anomaly*100:.1f}%)")

# Generate confusion matrix and analysis
cm_raw, cm_norm = plot_confusion_matrix(
    test_results['targets'], 
    test_results['predictions'], 
    class_names, 
    RESULTS_PATH
)

analyze_classification_errors(test_results['targets'], test_results['predictions'], class_names)


In [ ]:
# Save detailed results
def save_results(results, history, hyperparams, class_names, save_path):
    """Save all results to files"""
    
    # 1. Classification report (text format)
    report_text = classification_report(
        results['targets'], 
        results['predictions'], 
        target_names=class_names,
        digits=4
    )
    
    with open(save_path / 'classification_report_12class_deit.txt', 'w') as f:
        f.write("DeiT 12-Class Solar Panel Classification Report\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Model: DeiT-B/16 with custom classification head\n")
        f.write(f"Dataset: 12-class solar panel anomaly detection\n")
        f.write(f"Total test samples: {len(results['targets'])}\n")
        f.write(f"Overall accuracy: {results['accuracy']*100:.2f}%\n\n")
        f.write(report_text)
    
    # 2. Results summary (JSON format)
    results_summary = {
        'model_name': 'DeiT-B/16',
        'dataset': '12-class solar panel',
        'test_accuracy': float(results['accuracy']),
        'precision_macro': float(results['precision_macro']),
        'recall_macro': float(results['recall_macro']),
        'f1_macro': float(results['f1_macro']),
        'precision_weighted': float(results['precision_weighted']),
        'recall_weighted': float(results['recall_weighted']),
        'f1_weighted': float(results['f1_weighted']),
        'per_class_metrics': {
            class_names[i]: {
                'precision': float(results['per_class_precision'][i]),
                'recall': float(results['per_class_recall'][i]),
                'f1_score': float(results['per_class_f1'][i]),
                'support': int(results['per_class_support'][i])
            } for i in range(len(class_names))
        },
        'hyperparameters': hyperparams,
        'training_epochs': len(history['train_loss']),
        'best_val_accuracy': float(max(history['val_acc'])),
        'final_train_accuracy': float(history['train_acc'][-1]),
        'final_val_accuracy': float(history['val_acc'][-1])
    }
    
    with open(save_path / 'results_summary_12class_deit.json', 'w') as f:
        json.dump(results_summary, f, indent=2)
    
    # 3. Training history (JSON format)
    with open(save_path / 'training_history_12class_deit.json', 'w') as f:
        json.dump(history, f, indent=2)
    
    print(f"📁 Results saved to: {save_path}")
    print(f"  - Classification report: classification_report_12class_deit.txt")
    print(f"  - Results summary: results_summary_12class_deit.json")
    print(f"  - Training history: training_history_12class_deit.json")

# Inference functions
def predict_single_image(model, image_path, transform, device, class_names):
    """Predict single image"""
    model.eval()
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    input_tensor = transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        predicted_class_idx = outputs.argmax(1).item()
        confidence = probabilities[0][predicted_class_idx].item()
    
    predicted_class = class_names[predicted_class_idx]
    
    return {
        'predicted_class': predicted_class,
        'confidence': confidence,
        'all_probabilities': {class_names[i]: probabilities[0][i].item() for i in range(len(class_names))}
    }

def predict_batch_images(model, image_paths, transform, device, class_names, batch_size=32):
    """Predict batch of images"""
    model.eval()
    results = []
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        
        for path in batch_paths:
            image = Image.open(path).convert('RGB')
            batch_images.append(transform(image))
        
        batch_tensor = torch.stack(batch_images).to(device)
        
        with torch.no_grad():
            outputs = model(batch_tensor)
            probabilities = torch.nn.functional.softmax(outputs, dim=1)
            predicted_classes = outputs.argmax(1)
        
        for j, path in enumerate(batch_paths):
            predicted_class_idx = predicted_classes[j].item()
            confidence = probabilities[j][predicted_class_idx].item()
            
            results.append({
                'image_path': path,
                'predicted_class': class_names[predicted_class_idx],
                'confidence': confidence,
                'all_probabilities': {class_names[k]: probabilities[j][k].item() for k in range(len(class_names))}
            })
    
    return results

# Save all results
save_results(test_results, training_history, HYPERPARAMS, class_names, RESULTS_PATH)

# Example usage of inference functions
print("\n🔮 Inference Functions Created:")
print("  - predict_single_image(model, image_path, val_transform, device, class_names)")
print("  - predict_batch_images(model, image_paths, val_transform, device, class_names)")
print("\nExample usage:")
print("  result = predict_single_image(model, 'path/to/image.jpg', val_transform, device, class_names)")
print("  print(f'Predicted: {result[\"predicted_class\"]} (confidence: {result[\"confidence\"]:.3f})')")
